In [1]:
import os
import torch
import numpy as np
import pandas as pd
from typing import Dict, List
import time
from dotenv import load_dotenv

from langchain_core.messages import ChatMessage
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_text_splitters.base import TextSplitter

from financerag.common import get_query_and_retrieved_corpus_text, process_retrieval_df, get_final_result 
from financerag.task import *
from financerag.retrieval import DenseRetrieval
from financerag.hipporag import HippoRAG
from financerag.retrieval import BM25, BM25_Retriever
from financerag.rerank import CrossEncoderReranker

from sentence_transformers import CrossEncoder
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')


/Users/mac/Desktop/Code/Personal_Project/DeepLearningProject/RAG_Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(".env")
GG_API_KEY = os.environ.get('GOOGLE_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [3]:
def load_query(dataset_name : str, new_path_to_query = None) -> Dict[str, str]:
    query_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_queries.jsonl/queries.jsonl", lines = True)
    # Convert into Dict[str, str]
    query_dict = {row['_id'] : row['text'] for i, row in query_df.iterrows()}
    return query_dict

    
def load_corpus(dataset_name : str, new_path_to_corpus = None) -> Dict[str, str]:
    corpus_df = pd.read_json(f"finance_dataset/{dataset_name.lower()}_corpus.jsonl/corpus.jsonl", lines = True)
    # Convert into Dict[str, str]
    corpus_dict = {row['_id'] : row['text'] for i, row in corpus_df.iterrows()}
    return corpus_dict

In [4]:
# Set up vector database
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004", request_options = {'timeout' : 100000})
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [5]:
def get_vector_store():
    embeddings = GoogleGenerativeAIEmbeddings(model = "models/text-embedding-004",  request_options = {'timeout' : 100000})
    index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))
    
    vector_store = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
    )
    return vector_store

In [6]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [7]:
# Get dataset name first
import os
dataset_names = []
for f in os.listdir('finance_dataset'):
    if f.endswith("tsv"):
       dataset_names.append(f.split('_')[0])
dataset_names  

['MultiHeirtt',
 'FinQA',
 'FinanceBench',
 'ConvFinQA',
 'FinQABench',
 'TATQA',
 'FinDER']

In [8]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [12]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    print(f"{dataset_name} task")
    {task_variable} = {dataset_name}Task()
    {task_variable}.load()
    {task_variable}_max_len = np.max([len(text) for text in {task_variable}.corpus.values()])
    {task_variable}_top_k = ({task_variable}_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    print("Retrieve", {task_variable}_top_k, f"documents/query for {dataset_name} task")
    vector_store = get_vector_store()
    {task_variable}_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = {task_variable}.metadata.dataset_name)
    {task_variable}_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = {task_variable}.corpus, saved_index = False, batch_size = 500)

    {task_variable}_result = {task_variable}.retrieve(retriever = {task_variable}_retriever, top_k = {task_variable}_top_k)
    {task_variable}_reranker = CrossEncoderReranker(queries = {task_variable}.queries, corpus = {task_variable}.corpus, reranker = model)
    {task_variable}_final_result = {task_variable}_reranker.rerank(retrieved_result = {task_variable}_result, top_k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_document_and_reranking')
    """
    print(script_string)


    # MultiHeirtt Task
    print(f"MultiHeirtt task")
    multiheirtt_task = MultiHeirttTask()
    multiheirtt_task.load()
    multiheirtt_task_max_len = np.max([len(text) for text in multiheirtt_task.corpus.values()])
    multiheirtt_task_top_k = (multiheirtt_task_max_len // (CHUNK_SIZE - CHUNK_OVERLAP) + 1) * 10 + 10
    print("Retrieve", multiheirtt_task_top_k, f"documents/query for MultiHeirtt task")
    vector_store = get_vector_store()
    multiheirtt_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = multiheirtt_task.metadata.dataset_name)
    multiheirtt_task_retriever.load_corpus_with_splitting(text_splitter = text_splitter, corpus = multiheirtt_task.corpus, saved_index = False, batch_size = 500)

    multiheirtt_task_result = multiheirtt_task.retrieve(retriever = multiheirtt_task_retriever, top_k = multiheirtt_task_top_k)
    multiheirtt_task_reranker = CrossEncoderReranker(queries = multiheirtt_task.queries, corpus = multiheirtt_task.corpus, reranke

In [14]:
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device=device)

In [11]:
print(f"FinanceBench task")
financebench_task = FinanceBenchTask()
financebench_task.load()
hippo_rag = HippoRAG(retriever=vector_store)


FinanceBench task


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [14]:
corpus = {}
for i, (corpus_id, corpus_text) in enumerate(financebench_task.corpus.items()):
    corpus[corpus_id] = corpus_text
    if i == 10: break
corpus

{'dd2af2336': 'PEPSICO_2022_10K 6) Africa, Middle East and South Asia (AMESA), which includes all of our beverage and convenient food businesses in\nAfrica, the Middle East and South Asia; and\n7) Asia Pacific, Australia and New Zealand and China Region (APAC), which includes all of our beverage and convenient\nfood businesses in Asia Pacific, Australia and New Zealand, and China region.',
 'dd2acf5c0': 'BOEING_2022_10K We derive a significant portion of our revenues from a limited number of commercial airlines.',
 'dd2ad12e4': 'COCACOLA_2022_10K THE COCA-COLA COMPANY AND SUBSIDIARIES\nCONSOLIDATED STATEMENTS OF CASH FLOWS\n(In millions)\nYear Ended December 31,\n2022\n2021\n2020\nOperating Activities\n \n \nConsolidated net income\n$\n9,571 $\n9,804 $\n7,768 \nDepreciation and amortization\n1,260 \n1,452 \n1,536 \nStock-based compensation expense\n356 \n337 \n126 \nDeferred income taxes\n(122)\n894 \n(18)\nEquity (income) loss net of dividends\n(838)\n(615)\n(511)\nForeign currency ad

In [34]:
hippo_rag.offline_indexing(corpus = corpus, batch_size = 16)
hippo_rag.retrieve(queries = financebench_task.queries, corpus = corpus, top_k = 10)

Retrieving: 100%|██████████| 150/150 [01:34<00:00,  1.58it/s]


{'qd2ac917a': {'dd2af3272': 40.368421052631575,
  'dd2ac04a8': 0.0,
  'dd2ad7824': 0.0,
  'dd2ade854': 0.0,
  'dd2af9b04': 0.0,
  'dd2aeab5e': 0.0,
  'dd2b02da8': 0.0,
  'dd2ade412': 0.0,
  'dd2ad12e4': 0.0,
  'dd2acf5c0': 0.0},
 'qd2aeff00': {'dd2b02da8': 7.027027027027026,
  'dd2ade412': 5.972972972972972,
  'dd2ac04a8': 0.0,
  'dd2ad7824': 0.0,
  'dd2ade854': 0.0,
  'dd2af9b04': 0.0,
  'dd2aeab5e': 0.0,
  'dd2af3272': 0.0,
  'dd2ad12e4': 0.0,
  'dd2acf5c0': 0.0},
 'qd2acfe6c': {'dd2ade412': 23.013513513513512,
  'dd2ade854': 14.054054054054053,
  'dd2ad7824': 5.972972972972972,
  'dd2ac04a8': 0.0,
  'dd2af9b04': 0.0,
  'dd2aeab5e': 0.0,
  'dd2b02da8': 0.0,
  'dd2af3272': 0.0,
  'dd2ad12e4': 0.0,
  'dd2acf5c0': 0.0},
 'qd2ad28d8': {'dd2b02da8': 7.027027027027026,
  'dd2ade412': 5.972972972972972,
  'dd2ac04a8': 0.0,
  'dd2ad7824': 0.0,
  'dd2ade854': 0.0,
  'dd2af9b04': 0.0,
  'dd2aeab5e': 0.0,
  'dd2af3272': 0.0,
  'dd2ad12e4': 0.0,
  'dd2acf5c0': 0.0},
 'qd2ac61e6': {'dd2b02da8': 7

In [36]:
hippo_rag.ner_extractor(financebench_task.queries['qd2abbd7c'])

[]

In [35]:
financebench_task.queries['qd2abbd7c']

'Is 3M a capital-intensive business based on FY2022 data?'

In [24]:
g.get_eid('nommi', 'khai', error = False)

-1

In [30]:
g.vs['name']

['khai', 'nommi', 'thu']

In [16]:
from langchain_core.output_parsers import ListOutputParser
import re
class CustomizedListParser(ListOutputParser):
    def parse(self, text) -> List[List[List[str]]]:
        pattern = r"\[\s*(?:\[\s*(?:\[[^\]]*?\]\s*,?\s*\n*)*\s*\]\s*,?\s*)*\s*\]"
        result = re.findall(pattern, text)
        if result:
            extracted_text = re.findall(pattern, text)[0]
            return eval(extracted_text)
        return []

In [17]:
test_str = """

[
 [["Radio City", "located in", "India"],
 ["Radio City", "is", "private FM radio station"],
 ["Radio City", "started on", "3 July 2001"],
 ["Radio City", "plays songs in", "Hindi"],
 ["Radio City", "plays songs in", "English"],
 ["Radio City", "forayed into", "New Media"],
 ["Radio City", "launched", "PlanetRadiocity.com"],
 ["PlanetRadiocity.com", "launched in", "May 2008"],
 ["PlanetRadiocity.com", "is", "music portal"],
 ["PlanetRadiocity.com", "offers", "news"],
 ["PlanetRadiocity.com", "offers", "videos"],
 ["PlanetRadiocity.com", "offers", "songs"]]
 ]
 """

CustomizedListParser().parse(test_str * 100000)


[[['Radio City', 'located in', 'India'],
  ['Radio City', 'is', 'private FM radio station'],
  ['Radio City', 'started on', '3 July 2001'],
  ['Radio City', 'plays songs in', 'Hindi'],
  ['Radio City', 'plays songs in', 'English'],
  ['Radio City', 'forayed into', 'New Media'],
  ['Radio City', 'launched', 'PlanetRadiocity.com'],
  ['PlanetRadiocity.com', 'launched in', 'May 2008'],
  ['PlanetRadiocity.com', 'is', 'music portal'],
  ['PlanetRadiocity.com', 'offers', 'news'],
  ['PlanetRadiocity.com', 'offers', 'videos'],
  ['PlanetRadiocity.com', 'offers', 'songs']]]

In [15]:
for dataset_name in dataset_names:
    task_variable = f"{dataset_name.lower()}_task"
    script_string = f"""
    # {dataset_name} Task
    {task_variable}_final_result = extract_result({task_variable}_result, k = 10)
    {task_variable}.save_retrieved_results({task_variable}_final_result, method_name = 'dense_retrieval_split_only')
    """
    print(script_string)


    # MultiHeirtt Task
    multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
    multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinQA Task
    finqa_task_final_result = extract_result(finqa_task_result, k = 10)
    finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinanceBench Task
    financebench_task_final_result = extract_result(financebench_task_result, k = 10)
    financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # ConvFinQA Task
    convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
    convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')
    

    # FinQABench Task
    finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
  

In [21]:
def extract_result(task_result : Dict[str, Dict[str, float]], k = 10):
    final_result = {}
    for query_id, doc_dict in task_result.items():
        final_result[query_id] = {}
        for i, (corpus_id, score) in enumerate(doc_dict.items()):
            if (i == k): break
            final_result[query_id][corpus_id] = score
    return final_result

In [22]:

# # FinQA Task
# finqa_task_final_result = extract_result(finqa_task_result, k = 10)
# finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinanceBench Task
# financebench_task_final_result = extract_result(financebench_task_result, k = 10)
# financebench_task.save_retrieved_results(financebench_task_final_result, method_name = 'dense_retrieval_split_only')


# # ConvFinQA Task
# convfinqa_task_final_result = extract_result(convfinqa_task_result, k = 10)
# convfinqa_task.save_retrieved_results(convfinqa_task_final_result, method_name = 'dense_retrieval_split_only')


# # FinQABench Task
# finqabench_task_final_result = extract_result(finqabench_task_result, k = 10)
# finqabench_task.save_retrieved_results(finqabench_task_final_result, method_name = 'dense_retrieval_split_only')


# # TATQA Task
# tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
# tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_split_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_split_only')

Saved result successfully to ./financerag_result/dense_retrieval_split_only/finder_result.csv!


In [13]:





# TATQA Task
tatqa_task_final_result = extract_result(tatqa_task_result, k = 10)
tatqa_task.save_retrieved_results(tatqa_task_final_result, method_name = 'dense_retrieval_only')


# FinDER Task
finder_task_final_result = extract_result(finder_task_result, k = 10)
finder_task.save_retrieved_results(finder_task_final_result, method_name = 'dense_retrieval_only')

Saved result successfully to ./financerag_result/dense_retrieval_only/tatqa_result.csv!
Saved result successfully to ./financerag_result/dense_retrieval_only/finder_result.csv!


In [26]:
method_name = 'dense_retrieval_split_document_and_reranking'
final_result = get_final_result(dataset_names, method_name = method_name)

In [24]:
final_result

,query_id,corpus_id
0,q82d4c6ec,d8bcdd900
1,q82d4c6ec,d8177a896
2,q82d4c6ec,d8a3219a8
3,q82d4c6ec,d8e8a6550
4,q82d4c6ec,d87914156
...,...,...
2155,q00218,BRK.A20232361
2156,q00218,BRK.A20232245
2157,q00218,BRK.A20232333
2158,q00218,BRK.A20230921


In [27]:
final_result

,query_id,corpus_id
0,q82d4c6ec,d81a04fe4
1,q82d4c6ec,d8d83b68e
2,q82d4c6ec,d8d3fbbaa
3,q82d4c6ec,d8d569dde
4,q82d4c6ec,d88be204e
...,...,...
2155,q00218,BRK.A20232226
2156,q00218,BRK.A20231068
2157,q00218,BRK.A20231726
2158,q00218,BRK.A20231067


In [28]:
final_result.to_csv(f'submission_{method_name}.csv', index = False)

In [11]:
import torch.nn as nn

In [45]:

# FinQA Task
print(f"FinQA task")
finqa_task = FinQATask()
finqa_task.load()
vector_store = get_vector_store()
finqa_task_retriever = DenseRetrieval(vector_store = vector_store, dataset_name = finqa_task.metadata.dataset_name)
finqa_task_retriever.load_corpus_without_splitting(corpus = finqa_task.corpus, saved_index = False)
finqa_task_result = finqa_task.retrieve(retriever = finqa_task_retriever, top_k = 50)
finqa_task_reranker = CrossEncoderReranker(queries = finqa_task.queries, corpus = finqa_task.corpus, reranker = model)
finqa_task_final_result = finqa_task_reranker.rerank(finqa_task_result, top_k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_and_reranking')





FinQA task


Reranking: 100%|██████████| 1147/1147 [08:19<00:00,  2.29it/s]

Saved result successfully to ./financerag_result/dense_retrieval_and_reranking/finqa_result.csv!


In [46]:
# # MultiHeirtt Task
# multiheirtt_task_final_result = extract_result(multiheirtt_task_result, k = 10)
# multiheirtt_task.save_retrieved_results(multiheirtt_task_final_result, method_name = 'dense_retrieval_only')


# FinQA Task
finqa_task_final_result = extract_result(finqa_task_result, k = 10)
finqa_task.save_retrieved_results(finqa_task_final_result, method_name = 'dense_retrieval_only')





Saved result successfully to ./financerag_result/dense_retrieval_only/finqa_result.csv!


In [21]:
a = {'s': {'s' : 1}}
isinstance(a, Dict)

True